## Preparing the dataset for modelling (NOT READY YET)

### general paths and libraries

In [ ]:
!pip install seaborn

In [ ]:
import os
import numpy as np
import rasterio
from rasterio import mask
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# variables - Best results (pearson correlation) for window size
lst_clipped_path = '../../datos-notebooks/0.LST/LST_clipped_celsius.tif'
ndvi_1 = '../../datos-notebooks/1.land-cover/moving-windows/ndvi/Class_1_window25.tif'
ndwi_1 = '../../datos-notebooks/1.land-cover/moving-windows/ndwi/Class_1_window61.tif'
veg_vol = '../../datos-notebooks/2.objects-height/moving-windows/tree-volume/trees_window13.tif'
built_vol = '../../datos-notebooks/2.objects-height/moving-windows/building-volume/buildings_window15.tif'
built_area = '../../datos-notebooks/2.objects-height/moving-windows/building-area/building_area_window11.tif'
mean_elev = '../../datos-notebooks/3.topography/moving_windows/elev_mean/elev_mean_win111.tif'
mean_slope = '../../datos-notebooks/3.topography/moving_windows/slope_mean/slope_mean_win19.tif'
northness = '../../datos-notebooks/3.topography/moving_windows/northness/northness_win201.tif'

output_dir = '../../datos-notebooks/4.LST-prediction/'
os.makedirs(output_dir, exist_ok=True)

In [ ]:

rasters = {
    "LST": lst_clipped_path,
    "Dense vegetation": ndvi_1,
    "Water": ndwi_1,
    "Vegetation Volume": veg_vol,
    "Building Volume": built_vol,
    "Building Area": built_area,
    "Elevation": mean_elev,
    "Slope": mean_slope,
    "Northness": northness
}

n_vars = len(rasters)

with rasterio.open(lst_clipped_path) as src:
    lst = src.read(1)

lst_mask = np.isnan(lst)  # True where LST is NaN


fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, (name, path) in zip(axes.flat, rasters.items()):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)

    # Apply LST NaN mask to all rasters
    data[lst_mask] = np.nan

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

### stratified sampling of LST

In [ ]:
# Read and flatten
data_arrays = {}
for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        data_arrays[name] = arr.flatten()

df = pd.DataFrame(data_arrays)

# Drop rows with NaN
df = df.dropna()

# Create LST bins for stratification (5 bins)
n_bins = 10
df['LST_bin'] = pd.qcut(df['LST'], n_bins, labels=False)



In [ ]:
df['LST_bin'].value_counts().sort_index()


In [ ]:
# Stratified sampling: 5% of each bin
sample_fraction = 0.05
df_sampled = df.groupby('LST_bin', group_keys=False).apply(
    lambda x: x.sample(frac=sample_fraction, random_state=42)
)

# Drop bin column
df_sampled = df_sampled.drop(columns=['LST_bin']).reset_index(drop=True)

print(df_sampled.head())
print("Sampled dataset shape:", df_sampled.shape)

# Split features and target
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']


In [ ]:
# Number of rows and columns
print(df_sampled.shape)  
# Output: (number_of_pixels, number_of_variables)

# Number of pixels (rows)
print(len(df_sampled))

# Number of features (columns) excluding target
print(X.shape[1])

# Quick info about dataset
print(df_sampled.info())


In [ ]:
# Recreate bins for the sampled dataset
n_bins = 10
df_sampled['LST_bin'] = pd.qcut(df_sampled['LST'], n_bins, labels=False)

# Show min and max LST in each bin
for b in range(n_bins):
    values_in_bin = df_sampled[df_sampled['LST_bin'] == b]['LST']
    print(f"Bin {b}: min = {values_in_bin.min():.2f}, max = {values_in_bin.max():.2f}, count = {len(values_in_bin)}")



In [ ]:
# Compute correlation matrix (only features)
corr_matrix = X.corr()

# Display heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()


### Descriptive statistics

In [ ]:
for i, name in enumerate(rasters.keys()):
    plt.figure()
    plt.hist(X_valid[:, i], bins=50)
    plt.title(name)
    plt.show()


In [ ]:
lst_vals = df["LST"]

for v in df.columns:
    if v == "LST": continue
    plt.figure()
    plt.scatter(df[v], lst_vals, s=1, alpha=0.3)
    plt.xlabel(v)
    plt.ylabel("LST")
    plt.title(f"LST vs {v}")
    plt.show()


### PCA

In [ ]:
import rasterio
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---------------------------
# Load rasters
# ---------------------------
data_arrays = []
predictor_names = []  # names excluding LST
profile_ref = None

for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if name == "LST":
            lst = arr.copy()        # keep LST separate
            profile_ref = src.profile
        else:
            data_arrays.append(arr)
            predictor_names.append(name)

# Stack predictors to (n_pixels, n_variables)
stack = np.stack(data_arrays, axis=-1)
nrows, ncols, nbands = stack.shape
X = stack.reshape(-1, nbands)

# ---------------------------
# Mask invalid pixels (use LST as reference)
# ---------------------------
valid_mask = ~np.isnan(lst.reshape(-1))
X_valid = X[valid_mask]

# ---------------------------
# Standardize variables
# ---------------------------
scaler = StandardScaler()
X_std = scaler.fit_transform(X_valid)

# ---------------------------
# PCA
# ---------------------------
pca = PCA()
X_pca = pca.fit_transform(X_std)

# ---------------------------
# Explained variance
# ---------------------------
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative variance:", np.cumsum(pca.explained_variance_ratio_))

# ---------------------------
# Save first 4 PCs as rasters
# ---------------------------
for i in range(4):
    pc_full = np.full(X.shape[0], np.nan)
    pc_full[valid_mask] = X_pca[:, i]

    pc_img = pc_full.reshape(nrows, ncols)

    out_path = os.path.join(output_dir, f"PC{i+1}.tif")
    profile = profile_ref.copy()
    profile.update(dtype=rasterio.float32, count=1, compress="lzw")

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(pc_img.astype(np.float32), 1)

    print(f"Saved: {out_path}")

# ---------------------------
# Loadings (variable weights per PC)
# ---------------------------
print("\nLoadings (PC weights):")
for i, comp in enumerate(pca.components_):
    print(f"\nPC{i+1}")
    for name, weight in zip(predictor_names, comp):
        print(f"{name:12s}: {weight:.3f}")



In [ ]:
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.ylim(0,1)
plt.title("Cumulative variance")
plt.grid(True)
plt.show()


In [ ]:
for v in df.columns:
    plt.figure()
    plt.boxplot(df[v], vert=False)
    plt.title(v)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from itertools import combinations

# LST vector para los píxeles válidos
lst_vals = lst.reshape(-1)[valid_mask]

# Elegir PCs a graficar (1 a 4)
pcs_to_plot = X_pca[:, :4]
pc_labels = [f"PC{i+1}" for i in range(4)]

# Todas las combinaciones de 2 PCs
pc_combos = list(combinations(range(4), 2))
n_combos = len(pc_combos)

# Crear subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, (i, j) in zip(axes, pc_combos):
    sc = ax.scatter(pcs_to_plot[:, i], pcs_to_plot[:, j],
                    c=lst_vals, s=1, cmap="inferno", alpha=0.5)
    ax.set_xlabel(pc_labels[i])
    ax.set_ylabel(pc_labels[j])
    ax.set_title(f"{pc_labels[i]} vs {pc_labels[j]}")

# Colorbar común
plt.suptitle("PCA scatter plots colored by LST", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create DataFrame with first 4 PCs
loadings = pd.DataFrame(pca.components_[:4],
                        columns=predictor_names,
                        index=[f"PC{i+1}" for i in range(4)])

# Bar plot
loadings.T.plot(kind="bar", figsize=(12, 6))
plt.ylabel("Loading value")
plt.title("PCA Loadings (PC1–PC4)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### stepwise

In [ ]:
!pip install statsmodels
import numpy as np
import statsmodels.api as sm

In [ ]:


# Target variable (LST)
y = lst_vals  # already masked with valid_mask

# Predictors: first 6 PCs (can be changed)
X = X_pca[:, :9]

# Add constant (intercept)
X_full = sm.add_constant(X)

# Names
pc_names = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6"]

# Forward stepwise selection using AIC
selected = []
remaining = pc_names.copy()
current_aic = np.inf

while remaining:
    aic_with_candidates = []
    
    for candidate in remaining:
        vars_idx = [pc_names.index(v) for v in selected + [candidate]]
        X_test = X_full[:, [0] + [i+1 for i in vars_idx]]  # intercept + PCs
        model = sm.OLS(y, X_test).fit()
        aic_with_candidates.append((model.aic, candidate))
    
    aic_with_candidates.sort()
    best_aic, best_candidate = aic_with_candidates[0]
    
    if best_aic < current_aic:
        current_aic = best_aic
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        print(f"Added: {best_candidate}, AIC = {best_aic:.2f}")
    else:
        break

print("\nSelected PCs:", selected)

# Fit final model
final_idx = [pc_names.index(v) for v in selected]
X_final = X_full[:, [0] + [i+1 for i in final_idx]]
final_model = sm.OLS(y, X_final).fit()

print(final_model.summary())
